In [2]:
import gmsh
import sys

In [5]:

def generate_3d_fluid_mesh():
    gmsh.initialize()
    gmsh.model.add("OpenFOAM_3D_Unstructured_mesh")
    
    #Geometry parameters (fluid domain)
    L,W,H = 10,5,5 #(in metres)
    r= 0.5 #here, we are considering a sphere in the fluid domain
    cx,cy,cz = 3,2.5,2.5 #position of the centre of the sphere
    
    #create geometry in gmsh
    box= gmsh.model.occ.addBox(0,0,0,L,W,H)
    sphere= gmsh.model.occ.addSphere(cx,cy,cz,r)
    
    #fluid domain : box-sphere
    fluid_v,_ = gmsh.model.occ.cut([(3,box)],[(3,sphere)])
    gmsh.model.occ.synchronize()
    
    #Identify boundaries
    surfaces = gmsh.model.getBoundary(fluid_v, oriented = False)
    inlet, outlet, walls, object_surf = [],[],[],[]
    
    for dim, tag in surfaces:
        com = gmsh.model.occ.getCenterOfMass(dim, tag)
        if abs(com[0]-0)< 1e-3:
            inlet.append(tag)
        elif abs(com[0]-L)< 1e-3:
            outlet.append(tag)
        elif (cx-r-0.1< com[0]< cx+r+0.1) and (cy-r-0.1<com[1]<cy+r+0.1):
            object_surf.append(tag)
        else:
            walls.append(tag)
            
    # Physical groups
    gmsh.model.addPhysicalGroup(2, inlet,name = "inlet")
    gmsh.model.addPhysicalGroup(2, outlet, name="outlet")
    gmsh.model.addPhysicalGroup(2, walls, name="walls")
    gmsh.model.addPhysicalGroup(2, object_surf, name="object")
    
    fluid_tags =[tag for dim, tag in fluid_v]
    gmsh.model.addPhysicalGroup(3,fluid_tags, name= "internalfield")
    
    #Refinement fields
    f_dist = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers( f_dist, "SurfacesList", object_surf)
    
    f_thresh = gmsh.model.mesh.field.add("Threshold")
    gmsh.model.mesh.field.setNumber(f_thresh,"InField",f_dist)
    gmsh.model.mesh.field.setNumber(f_thresh,"SizeMin",0.05)
    gmsh.model.mesh.field.setNumber(f_thresh,"SizeMax",1)
    gmsh.model.mesh.field.setNumber(f_thresh,"DistMin",0.2)
    gmsh.model.mesh.field.setNumber(f_thresh,"DistMax",2.5)
    gmsh.model.mesh.field.setAsBackgroundMesh(f_thresh)
    
    #MeshOptions
    gmsh.option.setNumber("Mesh.Algorithm3D",10)
    gmsh.option.setNumber("Mesh.CharacteristicLengthExtendFromBoundary",0)
    
    
    #GenerateMesh
    gmsh.model.mesh.generate(3)
    gmsh.write("fluid_mesh_3d.msh")
    
        # 7. VISUALIZATION SETTINGS (need to check this again , used AI for visualization test)
    
    
    options = [
        ("Mesh.SurfaceFaces", 1),
        ("Mesh.Volume", 0),       
        ("Mesh.Volumes", 0),      
        ("Geometry.SurfaceAlpha", 40),
        ("General.AlphaBlending", 1)
    ]

    for opt, val in options:
        try:
            gmsh.option.setNumber(opt, val)
        except:
            print(f"[NOTE] Option {opt} not supported in this version. Skipping.")

    if '-nopopup' not in sys.argv:
        print("\n[SUCCESS] GUI Opening.")
        print("To see inside: Press 'a' on your keyboard to toggle transparency.")
        gmsh.fltk.run()


    
    gmsh.finalize()
    
    
    
if __name__ == "__main__":
    generate_3d_fluid_mesh()

Info    : Meshing 1D...
Info    : [ 10%] Meshing curve 14 (Circle)
Info    : [ 20%] Meshing curve 16 (Line)
Info    : [ 30%] Meshing curve 17 (Line)
Info    : [ 40%] Meshing curve 18 (Line)
Info    : [ 40%] Meshing curve 19 (Line)
Info    : [ 50%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 60%] Meshing curve 22 (Line)
Info    : [ 70%] Meshing curve 23 (Line)
Info    : [ 80%] Meshing curve 24 (Line)
Info    : [ 80%] Meshing curve 25 (Line)
Info    : [ 90%] Meshing curve 26 (Line)
Info    : [100%] Meshing curve 27 (Line)
Info    : Done meshing 1D (Wall 0.00203759s, CPU 0.002382s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 7 (Sphere, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 8 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 9 (Plane, Frontal-Delaunay)
Info    : [ 50%] Meshing surface 10 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 11 (Plane, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 12 (Plane, Front

Error   : Could not set option 'Mesh.Volume'
Error   : Could not set option 'Mesh.Volumes'
Error   : Could not set option 'Geometry.SurfaceAlpha'
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x0


-------------------------------------------------------
Version       : 4.12.1
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 20240331
Build host    : debian-build-farm
Build options : 64Bit ALGLIB Bamg Blossom DIntegration Dlopen DomHex Eigen Fltk GMP Gmm Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis Mpeg ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR Voro++ WinslowUntangler Zlib
FLTK version  : 1.3.8
OCC version   : 7.6.3
Packaged by   : nobody
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


In [6]:
generate_3d_fluid_mesh()

Info    : Meshing 1D...
Info    : [ 10%] Meshing curve 14 (Circle)
Info    : [ 20%] Meshing curve 16 (Line)
Info    : [ 30%] Meshing curve 17 (Line)
Info    : [ 40%] Meshing curve 18 (Line)
Info    : [ 40%] Meshing curve 19 (Line)
Info    : [ 50%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 60%] Meshing curve 22 (Line)
Info    : [ 70%] Meshing curve 23 (Line)
Info    : [ 80%] Meshing curve 24 (Line)
Info    : [ 80%] Meshing curve 25 (Line)
Info    : [ 90%] Meshing curve 26 (Line)
Info    : [100%] Meshing curve 27 (Line)
Info    : Done meshing 1D (Wall 0.00216525s, CPU 0.002539s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 7 (Sphere, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 8 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 9 (Plane, Frontal-Delaunay)
Info    : [ 50%] Meshing surface 10 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 11 (Plane, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 12 (Plane, Front

Error   : Could not set option 'Mesh.Volume'
Error   : Could not set option 'Mesh.Volumes'
Error   : Could not set option 'Geometry.SurfaceAlpha'
X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x0
